<a href="https://colab.research.google.com/github/WhoisMonesh/Colab-Usenet-Downloader/blob/main/Colab-Usenet-Downloader.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Usenet Downloader

Download files from Usenet (.nzb files) to Google Drive.


### 1. Mount Google Drive

In [0]:
from google.colab import drive
drive.mount('/content/drive')

### 2. Install Dependencies

In [0]:
!pip install sabyenc -q
print('Dependencies installed.')

### 3. Configuration

In [0]:
# === CONFIGURATION ===
SAVE_PATH = '/content/downloads/UsenetDownloader/'  # local temp dir
DRIVE_PATH = '/content/drive/My Drive/UsenetDownloader/'  # final destination

SERVER_HOST = ''  # Usenet server (e.g., news.yourprovider.com)
SERVER_PORT = 563  # Usenet server port (563 = SSL)
USERNAME = ''  # Usenet account username
PASSWORD = ''  # Usenet account password

KEEP_ALIVE = True  # prevent Colab timeout

# === END CONFIGURATION ===

### 4. Download

In [0]:
import os, time, shutil, glob, ssl
from google.colab import files

def format_bytes(n):
    for u in ['B', 'KB', 'MB', 'GB', 'TB']:
        if n < 1024: return f'{n:.1f} {u}'
        n /= 1024
    return f'{n:.1f} PB'

def yenc_decode(raw):
    """Decode yEnc encoded article body to binary bytes."""
    in_body = False
    data = bytearray()
    for line in raw.split(b'\n'):
        line = line.rstrip(b'\r')
        if line.startswith(b'=ybegin'):
            in_body = True
            continue
        if line.startswith(b'=yend'):
            in_body = False
            continue
        if not in_body:
            continue
        i = 0
        while i < len(line):
            if line[i] == 61:  # '=' escape char
                data.append(line[i + 1] - 64)
                i += 2
            else:
                data.append(line[i] - 42)
                i += 1
    return bytes(data)

def parse_nzb(xml_text):
    import xml.etree.ElementTree as ET
    root = ET.fromstring(xml_text)
    ns = {'nzb': 'http://www.newzbin.com/DTD/2003/nzb'}
    files = []
    for fe in root.findall('.//nzb:file', ns):
        subj = fe.get('subject', '')
        fname = subj.split('"')[1] if '"' in subj else f'file_{len(files)}'
        groups = [g.text for g in fe.findall('.//nzb:group', ns) if g.text]
        segments = []
        for seg in fe.findall('.//nzb:segment', ns):
            segments.append(seg.text.strip())
        files.append({'name': fname, 'segments': segments, 'groups': groups})
    return files

def main():
    import nntplib
    from IPython.display import display, HTML, Javascript

    if KEEP_ALIVE:
        display(Javascript('''
            function keepAlive(){
                var btn=document.querySelector("colab-connect-button");
                if(btn)btn.click()
            }
            setInterval(keepAlive,120000);
        '''))
        print('Keep-alive active')

    if not SERVER_HOST or not USERNAME or not PASSWORD:
        print('ERROR: Set SERVER_HOST, USERNAME, and PASSWORD in config')
        return

    # Upload or paste NZB
    nzb_xml = ''
    print('Upload your .nzb file...')
    uploaded = files.upload()
    if not uploaded:
        print('ERROR: No file uploaded')
        return
    nzb_xml = list(uploaded.values())[0].decode('utf-8', errors='ignore')
    fname_uploaded = list(uploaded.keys())[0]
    print(f'Loaded: {fname_uploaded}')

    begin = time.time()
    os.makedirs(SAVE_PATH, exist_ok=True)
    print(f'Save path: {SAVE_PATH} (local)')

    files_meta = parse_nzb(nzb_xml)
    total_files = len(files_meta)
    print(f'Files in NZB: {total_files}')

    progress_display = display(HTML('<pre>Connecting...</pre>'), display_id='dl-progress')

    context = ssl.create_default_context()
    server = nntplib.NNTP_SSL(SERVER_HOST, port=SERVER_PORT, context=context, user=USERNAME, password=PASSWORD)
    print(f'Connected to {SERVER_HOST}')

    for idx, fe in enumerate(files_meta, 1):
        fname = fe['name']
        bar = '#' * (idx * 50 // total_files) + '-' * (50 - idx * 50 // total_files)
        progress_display.update(HTML(f'<pre>Downloading ({idx}/{total_files}): |{bar}| {fname[:50]}</pre>'))
        file_data = bytearray()
        for mid in fe['segments']:
            try:
                resp = server.body(f'<{mid}>')
                article_bytes = b'\n'.join(resp[1]) if isinstance(resp[1], list) else resp[1][0]
                file_data.extend(article_bytes)
            except Exception as e:
                print(f'  Segment failed: {mid[:40]}... - {e}')
        binary = yenc_decode(bytes(file_data))
        if binary:
            fpath = os.path.join(SAVE_PATH, fname)
            with open(fpath, 'wb') as f:
                f.write(binary)
            sz = format_bytes(len(binary))
            print(f'  [{idx}/{total_files}] {fname} ({sz})')
        else:
            print(f'  [{idx}/{total_files}] {fname} - EMPTY')

    server.quit()

    end = time.time()
    elapsed = int(end - begin)
    print()
    print('=' * 50)
    print('COMPLETE')
    print(f'Elapsed: {elapsed // 60}m {elapsed % 60}s')

    items = glob.glob(os.path.join(SAVE_PATH, '*'))
    items = [f for f in items if os.path.isfile(f)]

    if len(items) > 1:
        import zipfile
        total_sz = sum(os.path.getsize(f) for f in items)
        processed = 0
        zpath = SAVE_PATH.rstrip('/').rstrip('\\') + '.zip'
        print(f'\nZipping {len(items)} files ({format_bytes(total_sz)})...')
        with zipfile.ZipFile(zpath, 'w', zipfile.ZIP_DEFLATED) as zf:
            for fpath in items:
                zf.write(fpath, os.path.basename(fpath))
                processed += os.path.getsize(fpath)
                pct = int(processed * 100 / total_sz) if total_sz else 0
                bar = '#' * (pct // 2) + '-' * (50 - pct // 2)
                progress_display.update(HTML(f'<pre>Zipping: |{bar}| {pct}% | {os.path.basename(fpath)}</pre>'))
        print(f'Zip: {os.path.basename(zpath)} ({format_bytes(os.path.getsize(zpath))})')
        display(HTML(f'<a href="{zpath}" download>Download ZIP</a>'))
        files.download(zpath)

    print(f'\nMoving to Drive...')
    os.makedirs(DRIVE_PATH, exist_ok=True)
    for f in os.listdir(SAVE_PATH):
        shutil.move(os.path.join(SAVE_PATH, f), os.path.join(DRIVE_PATH, f))
    print(f'Final: {DRIVE_PATH}')
    for item in sorted(os.listdir(DRIVE_PATH)):
        print(f'  {item}')
    print('=' * 50)

if __name__ == '__main__':
    main()